# Hypothesis Testing - Examples

Practical demonstrations of hypothesis testing for ML.

## Topics Covered
1. Hypothesis Testing Framework
2. One-Sample t-Test
3. Two-Sample t-Test
4. Paired t-Test
5. Chi-Square Test of Independence
6. ANOVA
7. Multiple Testing Correction
8. Power Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=6, suppress=True)

## 1. Hypothesis Testing Framework

- **H₀ (Null)**: No effect / default assumption
- **H₁ (Alternative)**: Effect exists / what we want to show
- **α (Significance level)**: P(reject H₀ | H₀ true) = Type I error rate
- **p-value**: P(data as extreme or more | H₀ true)

In [ ]:
print("HYPOTHESIS TESTING FRAMEWORK")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────┐
│                    Decision Matrix                          │
├─────────────────────────────────────────────────────────────┤
│                     │ H₀ True         │ H₀ False            │
├─────────────────────┼─────────────────┼─────────────────────┤
│ Reject H₀           │ Type I Error    │ ✅ Correct          │
│                     │ (α = 0.05)      │ (Power = 1-β)       │
├─────────────────────┼─────────────────┼─────────────────────┤
│ Fail to Reject H₀   │ ✅ Correct      │ Type II Error       │
│                     │ (1-α)           │ (β)                 │
└─────────────────────┴─────────────────┴─────────────────────┘
""")

print("\n💡 Key Concepts:")
print("  - p < α → Reject H₀ (statistically significant)")
print("  - p ≥ α → Fail to reject H₀ (not significant)")
print("  - 'Fail to reject' ≠ 'Accept H₀'")

## 2. One-Sample t-Test

Tests if sample mean differs from hypothesized value.

$t = \frac{\bar{X} - \mu_0}{s / \sqrt{n}}$

In [ ]:
print("ONE-SAMPLE t-TEST")
print("="*60)

np.random.seed(42)

# Scenario: Model accuracy claimed to be 85%. Is our model different?
claimed_accuracy = 85
our_accuracies = np.array([87.2, 86.5, 88.1, 85.9, 87.8, 86.2, 88.5, 87.0, 86.8, 87.5])

print(f"\nClaimed accuracy: {claimed_accuracy}%")
print(f"Our accuracies: mean = {our_accuracies.mean():.2f}%, std = {our_accuracies.std(ddof=1):.2f}%")

# Calculate manually
n = len(our_accuracies)
x_bar = our_accuracies.mean()
s = our_accuracies.std(ddof=1)
se = s / np.sqrt(n)
t_stat = (x_bar - claimed_accuracy) / se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))  # Two-tailed

print(f"\n📊 Manual Calculation:")
print(f"  t = ({x_bar:.2f} - {claimed_accuracy}) / ({s:.2f}/√{n})")
print(f"  t = {t_stat:.4f}")
print(f"  df = {n-1}")
print(f"  p-value (two-tailed) = {p_value:.6f}")

# Using scipy
t_stat_scipy, p_value_scipy = stats.ttest_1samp(our_accuracies, claimed_accuracy)
print(f"\n📊 scipy.stats.ttest_1samp:")
print(f"  t = {t_stat_scipy:.4f}, p = {p_value_scipy:.6f}")

alpha = 0.05
print(f"\n🎯 Decision (α = {alpha}):")
if p_value_scipy < alpha:
    print(f"  p = {p_value_scipy:.4f} < {alpha} → REJECT H₀")
    print("  Our model accuracy is significantly different from 85%!")
else:
    print(f"  p = {p_value_scipy:.4f} ≥ {alpha} → FAIL TO REJECT H₀")
    print("  No significant difference from claimed accuracy.")

## 3. Two-Sample t-Test (Independent)

Tests if two independent groups have different means.

$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{s_p^2(1/n_1 + 1/n_2)}}$

In [ ]:
print("TWO-SAMPLE t-TEST")
print("="*60)

np.random.seed(42)

# Scenario: Compare two models' F1 scores
model_A = np.random.normal(0.82, 0.03, 25)  # Model A runs
model_B = np.random.normal(0.85, 0.04, 25)  # Model B runs

print(f"\nModel A: mean = {model_A.mean():.4f}, std = {model_A.std(ddof=1):.4f}, n = {len(model_A)}")
print(f"Model B: mean = {model_B.mean():.4f}, std = {model_B.std(ddof=1):.4f}, n = {len(model_B)}")

# Welch's t-test (doesn't assume equal variances)
t_stat, p_value = stats.ttest_ind(model_A, model_B, equal_var=False)

print(f"\n📊 Welch's t-test (unequal variance):")
print(f"  t = {t_stat:.4f}")
print(f"  p-value = {p_value:.6f}")

# Effect size (Cohen's d)
pooled_std = np.sqrt((model_A.var(ddof=1) + model_B.var(ddof=1)) / 2)
cohens_d = (model_B.mean() - model_A.mean()) / pooled_std

print(f"\n📏 Effect Size (Cohen's d): {cohens_d:.4f}")
print("  Interpretation: |d| < 0.2 small, 0.2-0.8 medium, > 0.8 large")

alpha = 0.05
print(f"\n🎯 Decision (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.4f} < {alpha} → REJECT H₀")
    print("  Models have significantly different performance!")
    print(f"  Model B is better by ~{(model_B.mean()-model_A.mean())*100:.2f}%")
else:
    print(f"  p = {p_value:.4f} ≥ {alpha} → FAIL TO REJECT H₀")

## 4. Paired t-Test

Tests differences in paired/matched samples (same subjects, before/after).

In [ ]:
print("PAIRED t-TEST")
print("="*60)

np.random.seed(42)

# Scenario: Same dataset, before vs after hyperparameter tuning
n_datasets = 15
baseline = np.random.uniform(0.70, 0.85, n_datasets)
tuned = baseline + np.random.normal(0.03, 0.02, n_datasets)  # Small improvement + noise

print(f"\nBaseline: mean = {baseline.mean():.4f}")
print(f"Tuned:    mean = {tuned.mean():.4f}")
print(f"Difference: {(tuned - baseline).mean():.4f} ± {(tuned - baseline).std():.4f}")

# Paired t-test
t_stat, p_value = stats.ttest_rel(baseline, tuned)

print(f"\n📊 Paired t-test:")
print(f"  t = {t_stat:.4f}")
print(f"  p-value = {p_value:.6f}")

# Compare with unpaired (WRONG approach)
t_wrong, p_wrong = stats.ttest_ind(baseline, tuned)

print(f"\n⚠️ If we used unpaired t-test (wrong!):")
print(f"  t = {t_wrong:.4f}, p = {p_wrong:.6f}")

print(f"\n💡 Paired test is more powerful because:")
print("  - It removes between-subject variability")
print("  - Only focuses on within-subject changes")

alpha = 0.05
print(f"\n🎯 Decision (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.6f} < {alpha} → REJECT H₀")
    print("  Tuning provides significant improvement!")
else:
    print(f"  p = {p_value:.6f} ≥ {alpha} → FAIL TO REJECT H₀")

## 5. Chi-Square Test of Independence

Tests if two categorical variables are independent.

In [ ]:
print("CHI-SQUARE TEST OF INDEPENDENCE")
print("="*60)

# Scenario: Is model type related to deployment success?
#              Success  Failure
#   Neural Net    85      15
#   Tree-Based    70      30
#   Linear        45      55

observed = np.array([
    [85, 15],   # Neural Net
    [70, 30],   # Tree-Based
    [45, 55]    # Linear
])

print("\nObserved Contingency Table:")
print("              Success  Failure")
print(f"Neural Net      {observed[0,0]:3d}      {observed[0,1]:3d}")
print(f"Tree-Based      {observed[1,0]:3d}      {observed[1,1]:3d}")
print(f"Linear          {observed[2,0]:3d}      {observed[2,1]:3d}")

# Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print(f"\n📊 Chi-Square Test:")
print(f"  χ² = {chi2:.4f}")
print(f"  df = {dof}")
print(f"  p-value = {p_value:.6f}")

print(f"\n📊 Expected counts (if independent):")
print("              Success  Failure")
for i, name in enumerate(['Neural Net', 'Tree-Based', 'Linear    ']):
    print(f"{name}   {expected[i,0]:6.1f}   {expected[i,1]:6.1f}")

# Effect size (Cramer's V)
n = observed.sum()
min_dim = min(observed.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\n📏 Effect Size (Cramér's V): {cramers_v:.4f}")
print("  0.1 = small, 0.3 = medium, 0.5 = large")

alpha = 0.05
print(f"\n🎯 Decision (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.4f} < {alpha} → REJECT H₀")
    print("  Model type and deployment success are NOT independent!")

## 6. ANOVA (Analysis of Variance)

Tests if 3+ group means are all equal.

In [ ]:
print("ONE-WAY ANOVA")
print("="*60)

np.random.seed(42)

# Scenario: Compare 4 different optimizers
sgd = np.random.normal(0.82, 0.03, 20)
adam = np.random.normal(0.85, 0.03, 20)
rmsprop = np.random.normal(0.84, 0.03, 20)
adagrad = np.random.normal(0.81, 0.03, 20)

print("\nOptimizer Performance (F1 scores):")
for name, data in [('SGD', sgd), ('Adam', adam), ('RMSprop', rmsprop), ('AdaGrad', adagrad)]:
    print(f"  {name:8s}: mean = {data.mean():.4f}, std = {data.std():.4f}")

# One-way ANOVA
f_stat, p_value = stats.f_oneway(sgd, adam, rmsprop, adagrad)

print(f"\n📊 One-Way ANOVA:")
print(f"  F = {f_stat:.4f}")
print(f"  p-value = {p_value:.6f}")

alpha = 0.05
print(f"\n🎯 Decision (α = {alpha}):")
if p_value < alpha:
    print(f"  p = {p_value:.6f} < {alpha} → REJECT H₀")
    print("  At least one optimizer performs differently!")
    
    # Post-hoc: Tukey HSD
    from scipy.stats import tukey_hsd
    result = tukey_hsd(sgd, adam, rmsprop, adagrad)
    print(f"\n📊 Post-hoc Tukey HSD (pairwise comparisons):")
    labels = ['SGD', 'Adam', 'RMSprop', 'AdaGrad']
    for i in range(4):
        for j in range(i+1, 4):
            sig = "*" if result.pvalue[i,j] < 0.05 else ""
            print(f"  {labels[i]} vs {labels[j]}: p = {result.pvalue[i,j]:.4f} {sig}")
else:
    print(f"  p = {p_value:.6f} ≥ {alpha} → FAIL TO REJECT H₀")

## 7. Multiple Testing Correction

In [ ]:
print("MULTIPLE TESTING CORRECTION")
print("="*60)

# Scenario: Testing 20 features for significance
np.random.seed(42)
n_tests = 20
# Most are null (no effect), a few have real effects
p_values = np.concatenate([
    np.random.uniform(0.001, 0.05, 3),  # 3 real effects
    np.random.uniform(0.01, 0.99, 17)   # 17 null effects
])
np.random.shuffle(p_values)

print(f"\nTesting {n_tests} features, p-values:")
print(f"  {np.round(p_values, 4)}")

alpha = 0.05

# No correction
sig_none = np.sum(p_values < alpha)
print(f"\n📊 No correction (α = {alpha}):")
print(f"  Significant: {sig_none} features")
print(f"  ⚠️ Expected false positives: {n_tests * alpha:.1f}")

# Bonferroni correction
alpha_bonf = alpha / n_tests
sig_bonf = np.sum(p_values < alpha_bonf)
print(f"\n📊 Bonferroni (α' = {alpha_bonf:.4f}):")
print(f"  Significant: {sig_bonf} features")
print("  Conservative but controls FWER (family-wise error rate)")

# Benjamini-Hochberg (FDR)
sorted_p = np.sort(p_values)
ranks = np.arange(1, n_tests + 1)
bh_threshold = (ranks / n_tests) * alpha
sig_bh = np.sum(sorted_p <= bh_threshold)

print(f"\n📊 Benjamini-Hochberg (FDR control):")
print(f"  Significant: {sig_bh} features")
print("  Less conservative, controls FDR at α level")

print("\n💡 For ML feature selection:")
print("  - Bonferroni: Few false positives, may miss real effects")
print("  - B-H: Good balance, preferred in exploratory analysis")

## 8. Power Analysis

In [ ]:
print("POWER ANALYSIS")
print("="*60)

from scipy.stats import norm

# Parameters
alpha = 0.05
effect_size = 0.5  # Cohen's d

def calculate_power(n, alpha, effect_size):
    """Calculate power for two-sample t-test."""
    se = np.sqrt(2 / n)  # Standard error under H0
    z_alpha = norm.ppf(1 - alpha/2)
    z_power = effect_size / se - z_alpha
    return norm.cdf(z_power)

print(f"\nSettings: α = {alpha}, Effect size (d) = {effect_size}")
print(f"\n{'Sample Size (n)':>15} {'Power':>10}")
print("-"*30)

for n in [10, 20, 30, 50, 75, 100, 150, 200]:
    power = calculate_power(n, alpha, effect_size)
    marker = "← 80% power" if 0.78 < power < 0.82 else ""
    print(f"{n:>15} {power:>10.3f} {marker}")

# Find n for 80% power
from scipy.optimize import brentq

def power_minus_target(n):
    return calculate_power(n, alpha, effect_size) - 0.80

n_80 = brentq(power_minus_target, 5, 500)
print(f"\n✅ For 80% power: n ≈ {n_80:.0f} per group")

# Visualization
ns = np.arange(5, 200)
powers = [calculate_power(n, alpha, effect_size) for n in ns]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns, powers, 'b-', linewidth=2)
ax.axhline(0.8, color='g', linestyle='--', label='80% Power Target')
ax.axvline(n_80, color='r', linestyle=':', label=f'n = {n_80:.0f}')
ax.set_xlabel('Sample Size (n per group)')
ax.set_ylabel('Power')
ax.set_title(f'Power Analysis: d = {effect_size}, α = {alpha}')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.show()

print("\n💡 Planning ML experiments:")
print("  - Estimate expected effect size from pilot/literature")
print("  - Use power analysis to determine sample size")
print("  - Underpowered studies waste resources!")